# 02 — Modeling
## Telco Customer Churn

Objetivo: comparar modelos de classificação, selecionar o champion por validação cruzada, avaliar ranking, escolher threshold sem usar o holdout e interpretar o modelo final.


In [ ]:
from pathlib import Path
import sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, classification_report,
    ConfusionMatrixDisplay, f1_score, precision_score, recall_score,
    RocCurveDisplay, PrecisionRecallDisplay, roc_auc_score
)
from sklearn.model_selection import (
    StratifiedKFold, RandomizedSearchCV, cross_validate,
    cross_val_predict, train_test_split
)
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
SRC_PATH = Path('../src').resolve()
if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

from data_preprocessing import load_telco_data, split_xy, build_preprocessor
from metrics import ranking_metrics

RANDOM_STATE = 42


## 1. Carregamento e split


In [ ]:
DATA_PATH = Path('../data/raw/telco_customer_churn.xlsx')
df = load_telco_data(DATA_PATH)
X, y = split_xy(df)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

print('Treino:', X_train.shape, f'churn={y_train.mean():.2%}')
print('Teste :', X_test.shape, f'churn={y_test.mean():.2%}')


## 2. Pré-processamento


In [ ]:
preprocessor = build_preprocessor(X_train)
preprocessor


## 3. Comparação de modelos com e sem balanceamento


In [ ]:
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
spw = neg / pos

models = {
    'logistic': LogisticRegression(max_iter=2000, solver='liblinear', random_state=RANDOM_STATE),
    'logistic_balanced': LogisticRegression(max_iter=2000, solver='liblinear', class_weight='balanced', random_state=RANDOM_STATE),
    'random_forest': RandomForestClassifier(n_estimators=500, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1),
    'random_forest_balanced': RandomForestClassifier(n_estimators=500, min_samples_leaf=2, class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=-1),
    'xgboost': XGBClassifier(n_estimators=400, max_depth=3, learning_rate=0.05, subsample=0.9, colsample_bytree=0.9, eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1),
    'xgboost_balanced': XGBClassifier(n_estimators=400, max_depth=3, learning_rate=0.05, subsample=0.9, colsample_bytree=0.9, scale_pos_weight=spw, eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {'roc_auc':'roc_auc','pr_auc':'average_precision','precision':'precision','recall':'recall','f1':'f1'}

rows = []
for name, estimator in models.items():
    pipe = Pipeline([('preprocessor', clone(preprocessor)), ('model', estimator)])
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=1)
    row = {'model': name}
    for metric in scoring:
        row[f'{metric}_mean'] = scores[f'test_{metric}'].mean()
        row[f'{metric}_std'] = scores[f'test_{metric}'].std()
    rows.append(row)

comparison = pd.DataFrame(rows).sort_values('roc_auc_mean', ascending=False).reset_index(drop=True)
comparison


## 4. Tuning — Random Forest


In [ ]:
rf_pipeline = Pipeline([
    ('preprocessor', clone(preprocessor)),
    ('model', RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1))
])
rf_params = {
    'model__n_estimators':[300,500,800],
    'model__max_depth':[None,4,6,8,12],
    'model__min_samples_split':[2,5,10,20],
    'model__min_samples_leaf':[1,2,5,10],
    'model__max_features':['sqrt','log2',0.5],
    'model__class_weight':[None,'balanced','balanced_subsample'],
}
rf_search = RandomizedSearchCV(
    rf_pipeline, rf_params, n_iter=25, scoring='roc_auc', cv=cv,
    random_state=RANDOM_STATE, n_jobs=-1, verbose=1, refit=True
)
rf_search.fit(X_train, y_train)
print('Melhor ROC-AUC CV:', rf_search.best_score_)
rf_search.best_params_


## 5. Tuning — XGBoost


In [ ]:
xgb_pipeline = Pipeline([
    ('preprocessor', clone(preprocessor)),
    ('model', XGBClassifier(objective='binary:logistic', eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1))
])
xgb_params = {
    'model__n_estimators':[200,300,400,600,800],
    'model__max_depth':[2,3,4,5,6],
    'model__learning_rate':[0.01,0.03,0.05,0.08,0.10],
    'model__min_child_weight':[1,3,5,8],
    'model__subsample':[0.7,0.8,0.9,1.0],
    'model__colsample_bytree':[0.7,0.8,0.9,1.0],
    'model__gamma':[0.0,0.1,0.3],
    'model__reg_alpha':[0.0,0.01,0.1],
    'model__reg_lambda':[1.0,2.0,5.0],
    'model__scale_pos_weight':[1.0,spw],
}
xgb_search = RandomizedSearchCV(
    xgb_pipeline, xgb_params, n_iter=30, scoring='roc_auc', cv=cv,
    random_state=RANDOM_STATE, n_jobs=1, verbose=1, refit=True
)
xgb_search.fit(X_train, y_train)
print('Melhor ROC-AUC CV:', xgb_search.best_score_)
xgb_search.best_params_


## 6. Comparação final dos candidatos


In [ ]:
best_base_name = comparison.iloc[0]['model']
best_base_pipe = Pipeline([
    ('preprocessor', clone(preprocessor)),
    ('model', models[best_base_name])
])

candidates = {
    best_base_name: best_base_pipe,
    'random_forest_tuned': rf_search.best_estimator_,
    'xgboost_tuned': xgb_search.best_estimator_,
}

candidate_rows = []
for name, pipe in candidates.items():
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=1)
    candidate_rows.append({
        'model': name,
        'roc_auc_cv': scores['test_roc_auc'].mean(),
        'pr_auc_cv': scores['test_pr_auc'].mean(),
        'f1_cv': scores['test_f1'].mean(),
    })

final_comparison = pd.DataFrame(candidate_rows).sort_values('roc_auc_cv', ascending=False).reset_index(drop=True)
final_comparison


## 7. Seleção do champion


In [ ]:
champion_name = final_comparison.iloc[0]['model']
champion = candidates[champion_name]
print('Champion:', champion_name)


## 8. Threshold usando apenas treino (OOF)

O ponto de corte é escolhido com previsões *out-of-fold* do conjunto de treino. Assim, o holdout permanece reservado para avaliação final.


In [ ]:
oof_prob = cross_val_predict(
    champion, X_train, y_train, cv=cv, method='predict_proba', n_jobs=1
)[:,1]

threshold_rows = []
for threshold in np.arange(0.20, 0.81, 0.01):
    pred = (oof_prob >= threshold).astype(int)
    threshold_rows.append({
        'threshold': threshold,
        'precision': precision_score(y_train, pred, zero_division=0),
        'recall': recall_score(y_train, pred, zero_division=0),
        'f1': f1_score(y_train, pred, zero_division=0),
        'percentual_base_acionada': pred.mean(),
    })

threshold_table_train = pd.DataFrame(threshold_rows)
best_threshold_row = threshold_table_train.loc[threshold_table_train['f1'].idxmax()]
best_threshold_row


## 9. Avaliação final no holdout


In [ ]:
selected_threshold = float(best_threshold_row['threshold'])
champion.fit(X_train, y_train)
y_prob = champion.predict_proba(X_test)[:,1]
y_pred = (y_prob >= selected_threshold).astype(int)

test_metrics = {
    'roc_auc': roc_auc_score(y_test, y_prob),
    'pr_auc': average_precision_score(y_test, y_prob),
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred, zero_division=0),
    'recall': recall_score(y_test, y_pred, zero_division=0),
    'f1': f1_score(y_test, y_pred, zero_division=0),
    'threshold': selected_threshold,
}
pd.Series(test_metrics, name='holdout').to_frame()


In [ ]:
print(classification_report(y_test, y_pred, target_names=['Não Churn','Churn'], zero_division=0))


## 10. Matriz de confusão, ROC e Precision-Recall


In [ ]:
fig, ax = plt.subplots(figsize=(6,5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=['Não Churn','Churn'], ax=ax)
ax.set_title(f'Matriz de Confusão — threshold={selected_threshold:.2f}')
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(7,5))
RocCurveDisplay.from_predictions(y_test, y_prob, ax=ax)
ax.set_title('ROC Curve — Holdout')
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(7,5))
PrecisionRecallDisplay.from_predictions(y_test, y_prob, ax=ax)
ax.set_title('Precision-Recall Curve — Holdout')
plt.tight_layout(); plt.show()


## 11. Lift@K e Capture Rate@K


In [ ]:
ranking_table = ranking_metrics(y_test, y_prob)
ranking_table


## 12. SHAP


In [ ]:
prep = champion.named_steps['preprocessor']
estimator = champion.named_steps['model']
sample = X_test.sample(min(500, len(X_test)), random_state=RANDOM_STATE)
Xt = prep.transform(sample)
if hasattr(Xt, 'toarray'):
    Xt = Xt.toarray()
feature_names = prep.get_feature_names_out()
model_class = estimator.__class__.__name__.lower()

if 'xgb' in model_class or 'forest' in model_class:
    explainer = shap.TreeExplainer(estimator)
    shap_values = explainer.shap_values(Xt)
    if isinstance(shap_values, list):
        shap_values = shap_values[-1]
else:
    explainer = shap.LinearExplainer(estimator, Xt)
    shap_values = explainer.shap_values(Xt)

shap.summary_plot(shap_values, Xt, feature_names=feature_names, max_display=20)


# Conclusões da Modelagem

Foram avaliados Regressão Logística, Random Forest e XGBoost, incluindo versões com balanceamento.

A seleção do modelo foi realizada por ROC-AUC médio em validação cruzada estratificada com 5 folds, preservando o conjunto de teste para avaliação final.

No estudo inicial desta base, a **Regressão Logística sem balanceamento** apresentou o melhor ROC-AUC médio (aprox. 0,6289), superando Random Forest e XGBoost, inclusive após tuning.

O threshold operacional é escolhido no conjunto de treino utilizando previsões out-of-fold. Isso evita utilizar o holdout para decidir o ponto de corte.

Além das métricas tradicionais, Lift@K e Capture Rate@K são utilizados para avaliar a capacidade do ranking de priorizar clientes sob capacidade operacional limitada.

Os resultados indicam capacidade discriminatória moderada e sugerem que a principal limitação está no poder informacional das features disponíveis, e não apenas na escolha do algoritmo.

Como evolução, recomenda-se incorporar variáveis comportamentais e transacionais, calibrar probabilidades e validar ações de retenção por experimentação controlada ou uplift modeling.
